In [ ]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

Review should focus on the behavior encoded in code cells, not on notebook metadata churn. This notebook keeps that review loop small: run fast.ai style hints when desired, and print nbdev-style code diffs when comparing notebooks.

Review tools draw a line between source changes and notebook noise. `style_check` is useful before publishing exported code because it combines fast.ai style hints with notebook hygiene reports, while `diff_nb` is useful during agent edits because it ignores outputs and metadata unless metadata is the only thing that changed.


In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import subprocess as _subprocess
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell as _mk_cell, read_nb as _read_nb, write_nb as _write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.review import code_source

In [ ]:
print("code cell:", code_source(_mk_cell("answer = 42", cell_type="code")))
print("markdown cell:", code_source(_mk_cell("Some docs", cell_type="markdown")))

code cell: answer = 42
markdown cell: None


In [ ]:
#| export
import ast
import glob
import json
import os
import re
import subprocess
import sys
from collections import Counter
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.nbio import read_nb as _read_nb
from fastcore.script import Param, call_parse
from nbdev.diff import nbs_pair, source_diff

from nbskill.foundation import (
    _empty_failure_map, _failure_map_path, _load_failure_map, cell_class_names,
    cell_hash, cell_source, cli_error, cli_return, exported_py_path, file_hash,
    is_export_directive, is_exported_code_cell, none_if_string, tracked_call,
)

### Style feedback as a tool

`style_check` wraps the fast.ai style checker so it can be called from the CLI or MCP without each caller rebuilding command arguments or handling strict mode.

In [ ]:
#| export
def _style_check_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["style_check", path]
    if skip_folder_re: argv += ["--skip-folder-re", str(skip_folder_re)]
    if skip_path: argv += ["--skip-path", str(skip_path)]
    return argv


def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts


def _notebook_paths(path="."):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file() and pth.suffix == ".ipynb":
        candidates = [pth]
    else:
        candidates = []
    return sorted({candidate for candidate in candidates if _is_notebook_path(candidate)})


def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not is_export_directive(line))


def _parse_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return None


def _top_level_function_count(tree):
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in tree.body)


def _assert_count(tree):
    return sum(isinstance(node, ast.Assert) for node in ast.walk(tree))


def _test_function_count(tree):
    return sum(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body
    )


def _import_keys(tree):
    keys = []
    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".", 1)[0]
                keys.append(f"import {alias.name} as {local}")
        elif isinstance(node, ast.ImportFrom):
            module = "." * node.level + (node.module or "")
            for alias in node.names:
                local = alias.asname or alias.name
                keys.append(f"from {module} import {alias.name} as {local}")
    return keys

In [ ]:
#| export
def _problem(code, path, cell=None, detail="", severity="warning", source="nbskill", **kwargs):
    problem = {
        "code": code,
        "path": str(path),
        "severity": severity,
        "source": source,
        "detail": detail,
    }
    if cell is not None: problem["cell_id"] = getattr(cell, "id", "")
    problem.update({key: value for key, value in kwargs.items() if value is not None})
    return problem


def _format_problem(problem):
    cell = f" id={problem['cell_id']}" if problem.get("cell_id") else ""
    line = f" line={problem['line']}" if problem.get("line") else ""
    fields = []
    for key in ("scope", "symbol", "import_key", "cells", "semantic_types", "confidence", "hint", "exported_py_path"):
        if key in problem:
            value = problem[key]
            if isinstance(value, list): value = ", ".join(map(str, value))
            fields.append(f"{key}={value!r}" if key in {"symbol", "import_key", "exported_py_path"} else f"{key}={value}")
    suffix = " ".join(fields + ([problem.get("detail", "")] if problem.get("detail") else []))
    return f"- {problem['code']}: {problem['path']}{cell}{line} {suffix}".rstrip()


def _nbskill_info(obj):
    meta = getattr(obj, "metadata", {}) or {}
    return meta.get("nbskill") if isinstance(meta, dict) else None


def _validation_problem(code, path, cell=None, detail="", **kwargs):
    return _problem(code, path, cell, detail=detail, severity="error", source="nbskill-validation", **kwargs)


def _cell_metadata_validation_problems(nb_path, cell):
    info = _nbskill_info(cell)
    if not isinstance(info, dict):
        return [_validation_problem("missing-cell-nbskill-metadata", nb_path, cell)]
    problems = []
    expected_hash = cell_hash(cell, n=None)
    stored_hash = info.get("source_hash")
    if not stored_hash:
        problems.append(_validation_problem("missing-cell-source-hash", nb_path, cell))
    elif stored_hash != expected_hash:
        problems.append(_validation_problem("cell-source-hash-mismatch", nb_path, cell, f"expected {expected_hash[:12]}, stored {str(stored_hash)[:12]}"))
    expected_type = getattr(cell, "cell_type", None)
    if "cell_type" not in info:
        problems.append(_validation_problem("missing-cell-type", nb_path, cell))
    elif info.get("cell_type") != expected_type:
        problems.append(_validation_problem("cell-type-mismatch", nb_path, cell, f"expected {expected_type!r}, stored {info.get('cell_type')!r}"))
    semantic_types = info.get("semantic_types")
    if not isinstance(semantic_types, list) or not semantic_types:
        problems.append(_validation_problem("missing-cell-semantic-types", nb_path, cell))
    return problems


def _notebook_export_hash_problems(nb_path, nb):
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return []
    info = _nbskill_info(nb)
    if not py_path.exists():
        return [_validation_problem("exported-py-missing", nb_path, detail=f"missing {py_path}", exported_py_path=str(py_path))]
    if not isinstance(info, dict) or not info.get("exported_py_hash"):
        return [_validation_problem("missing-exported-py-hash", nb_path, exported_py_path=str(py_path))]
    actual = file_hash(py_path)
    stored = info.get("exported_py_hash")
    if stored != actual:
        detail = f"expected {actual[:12]}, stored {str(stored)[:12]}"
        return [_validation_problem("exported-py-hash-mismatch", nb_path, detail=detail, exported_py_path=str(py_path))]
    return []

skip_style_paths = "_proc __pycache__ src assets examples tests archive".split(" ")

def notebook_validation_problems(path="."):
    "Return invalid nbskill metadata problems for notebooks under `path`."
    problems = []
    for nb_path in _notebook_paths(path):
        if any(x in nb_path.parts for x in skip_style_paths): continue
        try: nb = _read_nb(nb_path)
        except FileNotFoundError: continue
        problems.extend(_notebook_export_hash_problems(nb_path, nb))
        for cell in nb.cells: problems.extend(_cell_metadata_validation_problems(nb_path, cell))
    return problems


def _notebook_style_problems(path="."):
    problems = notebook_validation_problems(path)
    duplicate_imports = {}
    for nb_path in _notebook_paths(path):
        nb = _read_nb(nb_path)
        imports_by_scope = {"exported": {}, "internal": {}}
        for cell in nb.cells:
            classes = cell_class_names(cell)
            if "unclean_cell" in classes:
                visible = [name for name in classes if name != "unclean_cell"]
                problems.append(_problem("unclean-cell", nb_path, cell, semantic_types=visible))
            tree = _parse_code_cell(cell)
            if tree is None: continue
            source_lines = _source_without_directives(cell_source(cell)).splitlines()
            line_count = len(source_lines)
            function_count = _top_level_function_count(tree)
            if function_count > 2:
                problems.append(_problem("large-cell", nb_path, cell, f"{function_count} top-level functions"))
            if line_count > 20:
                problems.append(_problem("large-cell", nb_path, cell, f"{line_count} non-directive lines"))
            if "test_cell" in classes:
                problem_count = _assert_count(tree) + _test_function_count(tree)
                if problem_count > 3:
                    problems.append(_problem("multi-problem-test", nb_path, cell, f"{problem_count} asserts/test functions; split into one-problem-at-a-time cells"))
            scope = "exported" if is_exported_code_cell(cell) else "internal"
            for key in _import_keys(tree):
                imports_by_scope[scope].setdefault(key, []).append(getattr(cell, "id", ""))
        for scope, imports in imports_by_scope.items():
            for key, ids in imports.items():
                if len(ids) > 1:
                    duplicate_imports.setdefault(str(nb_path), []).append((scope, key, ids))
    for nb_path, items in duplicate_imports.items():
        for scope, key, ids in items:
            problems.append(_problem("duplicate-import", nb_path, scope=scope, import_key=key, cells=ids))
    if _notebook_paths(path):
        from nbskill.graph import notebook_order_problems
        problems.extend(notebook_order_problems(path))
    return problems


def _notebook_style_problem_lines(path="."):
    return [_format_problem(problem) for problem in _notebook_style_problems(path)]


def _format_notebook_style_report(path="."):
    lines = _notebook_style_problem_lines(path)
    if not lines: return "Notebook style report: no notebook hygiene problems found."
    return "\n".join(["Notebook style report:", *lines])

In [ ]:
#| export
def _format_count_group(title, counts):
    if not counts: return [f"{title}: none"]
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [f"{title}: " + ", ".join(f"{tool}={count}" for tool, count in ordered)]


def _format_failure_event(event):
    kind = event.get("kind", "event")
    tool = event.get("tool", "unknown")
    path = event.get("path")
    detail = event.get("summary") or event.get("error") or ",".join(event.get("reasons", []))
    location = f" path={path}" if path else ""
    return f"- {kind}: {tool}{location} {detail}".rstrip()


def _global_usage_summary_data():
    path = _failure_map_path()
    data = _load_failure_map(path) if path.exists() else _empty_failure_map()
    problems = [event for event in data.get("events", []) if event.get("kind") in {"failure", "friction"}][-5:]
    return {
        "path": str(path),
        "exists": path.exists(),
        "counts": data.get("counts", {}),
        "recent_problems": problems,
    }


def _format_global_usage_summary(data=None):
    data = data or _global_usage_summary_data()
    if not data["exists"]: return f"Global nbskill usage: no records at {data['path']}"
    counts = data.get("counts", {})
    lines = [f"Global nbskill usage: {data['path']}"]
    lines += _format_count_group("usage", counts.get("usage", {}))
    lines += _format_count_group("failures", counts.get("failures", {}))
    lines += _format_count_group("friction", counts.get("friction", {}))
    problems = data.get("recent_problems", [])
    if problems:
        lines.append("recent problems:")
        lines.extend(_format_failure_event(event) for event in problems)
    else:
        lines.append("recent problems: none")
    return "\n".join(lines)


def _reset_global_usage_summary():
    path = _failure_map_path()
    try: path.unlink()
    except FileNotFoundError: pass
    except OSError: path.write_text(json.dumps(_empty_failure_map(), indent=2, sort_keys=True), encoding="utf-8")

In [ ]:
#| export
_CHKSTYLE_RE = re.compile(r"^# (?P<path>.*?):cell\[(?P<cell>[^\]]+)\]:(?P<line>\d+): (?P<detail>.*)$")


def _cap_text(text, max_output_chars=12000):
    text = text or ""
    if max_output_chars is None or len(text) <= max_output_chars:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - max_output_chars
    return {
        "text": f"{text[:max_output_chars].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


def _chkstyle_diagnostics(text, max_diagnostics=200):
    diagnostics = []
    for line in (text or "").splitlines():
        match = _CHKSTYLE_RE.match(line)
        if not match: continue
        detail = match.group("detail")
        hint = None
        if "(hint:" in detail:
            detail, hint = detail.split("(hint:", 1)
            hint = hint.rstrip(")").strip()
        diagnostics.append({
            "source": "chkstyle",
            "code": detail.strip().split(" (", 1)[0],
            "path": match.group("path"),
            "cell_id": match.group("cell"),
            "line": int(match.group("line")),
            "severity": "hint",
            "detail": detail.strip(),
            "hint": hint,
        })
        if max_diagnostics and len(diagnostics) >= max_diagnostics: break
    return diagnostics


def _problem_chart(diagnostics):
    def counts(key):
        return dict(sorted(Counter(str(item.get(key, "")) for item in diagnostics if item.get(key)).items()))
    return {
        "by_code": counts("code"),
        "by_severity": counts("severity"),
        "by_path": counts("path"),
        "by_source": counts("source"),
    }


def _fix_suggestions(diagnostics):
    fixes = []
    for item in diagnostics:
        if item.get("code") == "duplicate-import":
            fixes.append({
                "code": "duplicate-import",
                "path": item.get("path"),
                "cells": item.get("cells", []),
                "description": f"Remove repeated import {item.get('import_key')} from later cells after checking scope.",
                "automatic": False,
            })
    return fixes


def style_report(
    path: str = ".",  # File or folder to check
    chkstyle: dict | None = None,  # Captured chkstyle result to include
    max_output_chars: int = 12000,  # Maximum raw chkstyle text to keep in report
    max_diagnostics: int = 200,  # Maximum parsed chkstyle diagnostics
):
    "Return structured style diagnostics, problem chart, and global nbskill usage data."
    notebook_problems = _notebook_style_problems(path)
    usage = _global_usage_summary_data()
    chkstyle = chkstyle or {"status": 0, "output": ""}
    capped = _cap_text(chkstyle.get("output", ""), max_output_chars=max_output_chars)
    diagnostics = [
        *(_chkstyle_diagnostics(chkstyle.get("output", ""), max_diagnostics=max_diagnostics)),
        *notebook_problems,
    ]
    notebook_text = _format_notebook_style_report(path)
    usage_text = _format_global_usage_summary(usage)
    chkstyle_text = capped["text"].strip()
    text = "\n\n".join(chunk for chunk in [chkstyle_text, notebook_text, usage_text] if chunk)
    return {
        "path": str(path),
        "summary": {
            "notebook_problem_count": len(notebook_problems),
            "chkstyle_problem_count": len([item for item in diagnostics if item.get("source") == "chkstyle"]),
            "diagnostic_count": len(diagnostics),
            "recent_problem_count": len(usage.get("recent_problems", [])),
            "output_truncated": capped["truncated"],
            "output_chars": capped["chars"],
            "omitted_chars": capped["omitted_chars"],
        },
        "diagnostics": diagnostics[:max_diagnostics] if max_diagnostics else diagnostics,
        "problem_chart": _problem_chart(diagnostics),
        "notebook_problems": notebook_problems,
        "global_usage": usage,
        "chkstyle": {"status": chkstyle.get("status", 0), **capped},
        "fixes": _fix_suggestions(diagnostics),
        "text": text,
    }

In [ ]:
report = style_report("nbs/data/test_nbskill.ipynb")
assert "summary" in report
assert "notebook_problems" in report
assert isinstance(report["notebook_problems"], list)
assert "Global nbskill usage:" in report["text"]

The structured report is meant for tools as much as people. `summary` gives a compact count of problems, `problem_chart` groups diagnostics by source and code, and `text` is the human-readable report that the CLI prints.

In [ ]:
sample_chkstyle = {
    "status": 1,
    "output": "# demo.ipynb:cell[abc123]:2: Missing whitespace (hint: add a blank line)",
}
sample_report = style_report(
    "nbs/data/test_nbskill.ipynb",
    chkstyle=sample_chkstyle,
    max_output_chars=200,
    max_diagnostics=5,
)
print(sample_report["summary"])
print(sample_report["problem_chart"]["by_source"])
print(sample_report["diagnostics"][0])

{'notebook_problem_count': 6, 'chkstyle_problem_count': 1, 'diagnostic_count': 7, 'recent_problem_count': 5, 'output_truncated': False, 'output_chars': 72, 'omitted_chars': 0}
{'chkstyle': 1, 'nbskill': 6}
{'source': 'chkstyle', 'code': 'Missing whitespace', 'path': 'demo.ipynb', 'cell_id': 'abc123', 'line': 2, 'severity': 'hint', 'detail': 'Missing whitespace', 'hint': 'add a blank line'}


In [ ]:
#| export
def run_style_check(path=".", skip_folder_re=None, skip_path=None, strict=False, max_output_chars=None):
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        status = _chkstyle_main(_style_check_argv(path, skip_folder_re, skip_path))
    output = "\n".join(chunk.rstrip() for chunk in (out.getvalue(), err.getvalue()) if chunk)
    capped = _cap_text(output, max_output_chars=max_output_chars) if max_output_chars else {"text": output, "truncated": False, "chars": len(output), "omitted_chars": 0}
    return {"status": status, "output": output, **capped}

In [ ]:
#| export
def _normalize_style_check_cli_aliases():
    aliases = {
        "--delete-after-output": "--delete_after_output",
        "--delete-after-outout": "--delete_after_outout",
    }
    sys.argv[:] = [aliases.get(arg, arg) for arg in sys.argv]


_normalize_style_check_cli_aliases()


@call_parse
@tracked_call
def style_check(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints or notebook hygiene problems are found
    delete_after_output: bool = False,  # Reset ~/.nbskill-errors.json after printing the global summary
    delete_after_outout: bool = False,  # Backward-compatible typo alias for delete_after_output
    max_output_chars: int = 12000,  # Cap printed chkstyle output
    max_diagnostics: int = 200,  # Cap returned diagnostics
    fix: bool = False,  # Show conservative fix suggestions
    dry_run: bool = True,  # Keep fix mode non-mutating by default
):
    "Print capped fast.ai style hints, notebook hygiene warnings, and global tool usage."
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    report = style_report(path, chkstyle=chkstyle, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    print(report["text"])
    if fix:
        print("\nFix suggestions:")
        fixes = report.get("fixes", [])
        if not fixes: print("- no deterministic fixes available")
        for item in fixes:
            mode = "would apply" if dry_run else "manual-review-required"
            print(f"- {mode}: {item['description']} ({item['path']})")
    if delete_after_output or delete_after_outout: _reset_global_usage_summary()
    has_problems = bool(report["diagnostics"])
    if strict and (chkstyle["status"] or has_problems): raise SystemExit(chkstyle["status"] or 1)
    return cli_return(chkstyle["status"] or int(has_problems))


@call_parse
@tracked_call
def validate_nbs(
    path: Param("Notebook file, folder, or glob to validate", str, opt=False, nargs="?") = "nbs",  # Notebook file, folder, or glob to validate
    strict: bool = True,  # Exit non-zero when invalid metadata is found
):
    "Validate nbskill metadata needed for safe notebook tools."
    problems = notebook_validation_problems(path)
    if problems:
        print("Notebook validation errors:")
        for problem in problems:
            print(_format_problem(problem))
        if strict: raise SystemExit(1)
    else:
        print("Notebook validation: no invalid nbskill metadata found.")
    return cli_return(int(bool(problems)))


def code_source(cell): return cell.source if cell.cell_type == "code" else None

`style_check` is the CLI-shaped wrapper around `style_report`. It captures fast.ai style output, appends notebook hygiene findings, prints the combined text report, and exits non-zero in strict mode when diagnostics are present.

In [ ]:
style_output = _StringIO()
with _redirect_stdout(style_output):
    style_check(
        "nbs/data/test_nbskill.ipynb",
        strict=False,
        max_output_chars=500,
        max_diagnostics=5,
    )
print("\n".join(style_output.getvalue().splitlines()[:6]))

# nbs/data/test_nbskill.ipynb:cell[032d561b]:5: closing bracket on its own line (hint: move `)`, `]`, or `}` to the end of the previous content line)
)
# nbs/data/test_nbskill.ipynb:cell[032d561b]:17: closing bracket on its own line (hint: move `)`, `]`, or `}` to the end of the previous content line)
)
# nbs/data/test_nbskill.ipynb:cell[4813dd41]:18: if single-statement body not one-liner (hint: put the simple body statement on the header line if it still reads clearly)
        if (folder / "py


In [ ]:
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import demo_path, remove_demo_path, stamp_notebook_metadata
from nbskill.review import notebook_validation_problems, validate_nbs

valid_path = demo_path("04_review_validate_valid.ipynb")
valid_nb = stamp_notebook_metadata(_new_nb([_mk_cell("assert True", cell_type="code")]))
_write_test_nb(valid_nb, valid_path)
assert notebook_validation_problems(valid_path) == []
validate_nbs(str(valid_path), strict=False)
remove_demo_path(valid_path)

In [ ]:
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import demo_path, remove_demo_path, stamp_notebook_metadata
from nbskill.review import notebook_validation_problems

invalid_path = demo_path("04_review_validate_invalid.ipynb")
invalid_nb = stamp_notebook_metadata(_new_nb([
    _mk_cell("x = 1", cell_type="code"),
    _mk_cell("Some docs", cell_type="markdown"),
    _mk_cell("assert True", cell_type="code"),
]))
del invalid_nb.cells[0].metadata["nbskill"]["source_hash"]
invalid_nb.cells[1].metadata["nbskill"]["semantic_types"] = []
invalid_nb.cells[2].metadata["nbskill"]["cell_type"] = "markdown"
_write_test_nb(invalid_nb, invalid_path)
codes = {problem["code"] for problem in notebook_validation_problems(invalid_path)}
assert {"missing-cell-source-hash", "missing-cell-semantic-types", "cell-type-mismatch"} <= codes
remove_demo_path(invalid_path)

In [ ]:
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb
from fastcore.nbio import write_nb as _write_test_nb
from nbskill.foundation import demo_path, remove_demo_path, stamp_notebook_metadata
from nbskill.review import notebook_validation_problems

export_nb_path = demo_path("04_review_export_hash.ipynb", base="nbs")
export_nb = stamp_notebook_metadata(_new_nb([_mk_cell("#| default_exp sample_tool", cell_type="code")]))
export_nb.metadata["nbskill"] = {"exported_py_hash": "bad"}
_write_test_nb(export_nb, export_nb_path)
codes = {problem["code"] for problem in notebook_validation_problems(export_nb_path)}
assert "exported-py-hash-mismatch" in codes
remove_demo_path(export_nb_path)

`notebook_validation_problems` is narrower than `style_report`: it only checks the nbskill metadata that makes guarded notebook edits reliable. `validate_nbs` prints the same failures for CLI use.

In [ ]:
validation_path = demo_path("04_review_validation_example.ipynb")
try:
    validation_nb = _new_nb([
        _mk_cell("x = 1", cell_type="code"),
        _mk_cell("Some docs", cell_type="markdown"),
    ])
    _write_test_nb(validation_nb, validation_path)
    problems = notebook_validation_problems(validation_path)
    print([problem["code"] for problem in problems[:4]])
finally:
    remove_demo_path(validation_path)

['missing-cell-nbskill-metadata', 'missing-cell-nbskill-metadata']


### Code-cell diffs

Notebook diffs are noisy when metadata and outputs are included. `diff_nb` asks nbdev for code-cell source on each side of a comparison and prints only the added, changed, or deleted code blocks the caller requested.

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return f"No git repository found for {str(path)!r}."
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass --ref_a None to compare against the working tree."
    )


def _git_root_rel(path):
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0: return None, None
    root = Path(root_cmd.stdout.strip())
    try: return root, path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return None, None


def _notebook_json_at_ref(path, ref):
    path = Path(path)
    if ref is None:
        return json.loads(path.read_text(encoding="utf-8"))
    root, rel = _git_root_rel(path)
    if root is None: return None
    show = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    if show.returncode != 0: return None
    return json.loads(show.stdout)


def _nbskill_metadata_by_cell(nb_json):
    cells = (nb_json or {}).get("cells", [])
    return {
        cell.get("id", str(idx)): (cell.get("metadata", {}) or {}).get("nbskill")
        for idx, cell in enumerate(cells)
    }


def _nbskill_metadata_change_count(path, ref_a, ref_b):
    try:
        old = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_a))
        new = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_b))
    except (OSError, json.JSONDecodeError, TypeError):
        return 0
    keys = set(old) | set(new)
    return sum(1 for key in keys if old.get(key) != new.get(key) and (old.get(key) is not None or new.get(key) is not None))


def _metadata_summary(count):
    if not count: return ""
    noun = "cell" if count == 1 else "cells"
    return f"Ignored nbskill metadata changes in {count} {noun}."


@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
):
    "Print nbdev-style diffs for code cells only; summarize nbskill metadata-only changes."
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
        cli_error(msg)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception as exc:
        detail = str(exc)
        hint = (
            f"Could not diff {path!r} against {ref_a!r}. "
            "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
            "Commit the notebook first, or pass --ref_a None to compare against the working tree."
        )
        if detail: hint += f"\nUnderlying error: {detail}"
        cli_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
    if text and metadata_summary: report = f"{text}\n\n{metadata_summary}"
    elif text: report = text
    elif metadata_summary: report = f"No code cell changes\n{metadata_summary}"
    else: report = "No code cell changes"
    print(report)
    return cli_return(report)

In [ ]:
path = demo_path("04_review_no_git.ipynb")
try:
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    try:
        diff_nb(str(path))
    except (SystemExit, ValueError) as exc:
        if isinstance(exc, SystemExit): assert exc.code == 1
        else:
            msg = str(exc)
            assert "No git repository" in msg or "Could not find notebook" in msg
finally:
    remove_demo_path(path)

root = demo_path("04_review_git")
try:
    root.mkdir()
    path = root / "demo.ipynb"
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    _subprocess.run(["git", "init"], cwd=root, check=True, capture_output=True)
    _subprocess.run(["git", "add", "demo.ipynb"], cwd=root, check=True, capture_output=True)
    _subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=root, check=True, capture_output=True)
    nb = _read_nb(path)
    nb.cells[0].metadata["nbskill"] = {"cell_type": "code", "semantic_types": [], "source_hash": "demo"}
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        diff_nb(str(path))
    text = out.getvalue()
    assert "No code cell changes" in text
    assert "Ignored nbskill metadata changes in 1 cell" in text
finally:
    remove_demo_path(root)

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import os as _os

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.foundation import remove_demo_path
from nbskill.review import (
    _format_global_usage_summary, _format_notebook_style_report, style_check,
)

path = demo_path("04_review_style.ipynb")
remove_demo_path(path)
try:
    long_source = "\n".join([f"x{i} = {i}" for i in range(21)])
    nb = new_nb([
        mk_cell("#| export\ndef a():\n    pass\ndef b():\n    pass\ndef c():\n    pass", cell_type="code"),
        mk_cell(long_source, cell_type="code"),
        mk_cell("assert 1 == 1\nassert 2 == 2\nassert 3 == 3\nassert 4 == 4", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("result = later_helper()", cell_type="code"),
        mk_cell("def later_helper():\n    return 1", cell_type="code"),
        mk_cell("def loader():\n    return MissingPath('x')", cell_type="code"),
    ])
    _write_nb(nb, path)
    report = _format_notebook_style_report(path)
    assert "large-cell" in report
    assert "multi-problem-test" in report
    assert "unclean-cell" in report
    assert "scope=exported" in report
    assert "scope=internal" in report
    assert "cell-order" in report
    assert "missing-import" in report

    custom_map = demo_path("04_review_errors.json")
    old_map = _os.environ.get("NBSKILL_FAILURE_MAP")
    try:
        _os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
        with _redirect_stdout(_StringIO()):
            style_check(str(path), delete_after_output=True)
        assert "usage:" in _format_global_usage_summary()
        assert not custom_map.exists()
    finally:
        if old_map is None:
            _os.environ.pop("NBSKILL_FAILURE_MAP", None)
        else:
            _os.environ["NBSKILL_FAILURE_MAP"] = old_map

    try:
        with _redirect_stdout(_StringIO()):
            style_check(str(path), strict=True)
    except SystemExit as exc:
        assert exc.code
    else:
        raise AssertionError("strict style_check should exit for notebook hygiene problems")
finally:
    remove_demo_path(path)


In [ ]:
assert code_source(_mk_cell("plain docs", cell_type="markdown")) is None
assert code_source(_mk_cell("answer = 42", cell_type="code")) == "answer = 42"

`diff_nb` deliberately compares code-cell source only. Markdown edits and notebook metadata churn stay out of the main diff so reviewers can see the executable behavior that changed.

In [ ]:
diff_root = demo_path("04_review_diff_example")
remove_demo_path(diff_root)
try:
    diff_root.mkdir()
    diff_path = diff_root / "demo.ipynb"
    _write_nb(new_nb([
        mk_cell("value = 1\nvalue", cell_type="code"),
        mk_cell("Original note", cell_type="markdown"),
    ]), diff_path)
    _subprocess.run(["git", "init"], cwd=diff_root, check=True, capture_output=True)
    _subprocess.run(["git", "add", "demo.ipynb"], cwd=diff_root, check=True, capture_output=True)
    _subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=diff_root, check=True, capture_output=True)

    diff_nb_json = _read_nb(diff_path)
    diff_nb_json.cells[0].source = "value = 2\nvalue"
    diff_nb_json.cells[1].source = "Updated note that will not appear in the code diff"
    _write_nb(diff_nb_json, diff_path)

    diff_output = _StringIO()
    with _redirect_stdout(diff_output):
        diff_nb(str(diff_path), ref_a="HEAD")
    print(diff_output.getvalue())
finally:
    remove_demo_path(diff_root)

--- code cell 43b8302d ---
--- 
+++ 
@@ -1,2 +1,2 @@
-value = 1
+value = 2
 value

